## Building Unit Cleaning Code, for San Diego County

Used for joining with SDGE utility

In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [2]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [ ]:
# load parcels 
parcels = gpd.read_parquet("/../../capstone/electrigrid/data/parcels/parcels_final.parquet").to_crs(epsg=4326)

# filter for San Diego
parcels = parcels[parcels['County'] == "San Diego"]

In [6]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

Building Footprint

Access: https://sat-io.earthengine.app/view/gba

In [7]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

In [ ]:
# load in SDG&E boundary map
sdge_shape = gpd.read_file("/../../capstone/electrigrid/data/utilities/IOU_shapefiles.geojson")

sdge_shape = sdge_shape[sdge_shape['Acronym'] == "SDG&E"]

## Unit-regression for multi-family homes

### Select parcels for multi-family homes

In [8]:
## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

### Select buildings that are multi-family homes and add Zillow data

In [9]:

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    predicate = "intersects")

building_zillow = building_zillow.drop(columns = "index_right")

In [48]:
parcels_res.head()

,PARNO,County,Shape_Length,Shape_Area,geometry
7629947,4241720700,San Diego,98.936758,434.984710,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,..."
7630042,4241721300,San Diego,76.116160,347.815384,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,..."
7630043,4241722000,San Diego,106.557535,579.922629,"MULTIPOLYGON Z (((-117.23549 32.79699 0.00000,..."
7630075,4236921300,San Diego,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,..."
7630078,4236220200,San Diego,67.240870,224.046738,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,..."


### Calculate the volume and aggregate based on parcels
The problem I think that's happening here is that the units are added to every single building. When multiple buildings are in one parcel the units are double counted. We need to add in the parcel number and either aggregate to the parcel.

In [11]:
# add in the parcel number to add all of the building volumes for the regression
building_zillow_parcels = building_zillow.sjoin(
                                parcels,
                                how = "left",
                                predicate = "intersects")

# drop the unnecessary columns
building_zillow_parcels = building_zillow_parcels.drop(columns = ['index_right', 'ADDRESS', 'CITY', 'ZIP','Shape_Length', 'Shape_Area'])

In [47]:
building_zillow_parcels.head()

,source,id,height,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-116.99693 33.29913, -116.99687 33.2...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-116.98655 33.25268, -116.98620 33.2...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-116.37224 33.25284, -116.37224 33.2...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego


In [12]:
# reproject data frame to crs with meters as units
building_m = building_zillow_parcels.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

In [46]:
building_m.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County,area_m2,volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego,412.421351,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego,452.260448,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego,232.343850,905.530915
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego,232.343850,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego,284.844083,1177.213436


In [14]:
# keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

Aggregate the volumes for the unit regression by parcel.

In [15]:
# aggregate the volumes by parcel
total_volume_m3 = building_w_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

In [16]:
# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'PARNO')

In [45]:
building_w_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County,area_m2,volume_m3,total_volume_m3,residual
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego,412.421351,1606.752247,1606.752247,-34.103414
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego,452.260448,1858.907171,1858.907171,-34.197265
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego,232.343850,905.530915,905.530915,-33.842424
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego,232.343850,905.530915,905.530915,-33.842424
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego,284.844083,1177.213436,1177.213436,-33.943542


The parcel data is from 2014 while the Zillow data is from 2025. There are some Zillow points not in parcels which is assumed to be due to the differences in time frames from the data. Parcels were likely added after 2014. 

In [17]:
# some of the homes don't have a parcel number (so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

In [44]:
building_w_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County,area_m2,volume_m3,total_volume_m3,residual
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego,412.421351,1606.752247,1606.752247,-34.103414
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego,452.260448,1858.907171,1858.907171,-34.197265
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego,232.343850,905.530915,905.530915,-33.842424
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego,232.343850,905.530915,905.530915,-33.842424
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego,284.844083,1177.213436,1177.213436,-33.943542


In [19]:
# remove duplicates from the parcel number so that it doesn't skew the linear regression
building_w_units = building_w_units.drop_duplicates(subset = 'PARNO', keep = 'first')

### Run the linear regression, remove outliers, and re-run the regression

In [20]:
# run model
results = smf.ols('unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

In [43]:
building_w_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County,area_m2,volume_m3,total_volume_m3,residual
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego,412.421351,1606.752247,1606.752247,-34.103414
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego,452.260448,1858.907171,1858.907171,-34.197265
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego,232.343850,905.530915,905.530915,-33.842424
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego,232.343850,905.530915,905.530915,-33.842424
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego,284.844083,1177.213436,1177.213436,-33.943542


In [22]:
# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

In [23]:
print(f"There are ", building_units_clean.shape[0], "buildings included in the regression.")
print(f"There are ", building_outliers.shape[0], "outliers")

There are  40482 buildings included in the regression.
There are  2777 outliers


In [24]:
# rerun linear regression
results_clean = smf.ols('unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

/tmp/ipykernel_1983840/26689504.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_1983840/26689504.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


In [25]:
intercept

18.124667213836098

In [26]:
slope

0.0004066340817752588

### Use the values from the linear regression to compute missing units

In [ ]:
# extract just the multi-family homes where unit info is missing
missing_units = building_m[building_m['unit'].isna()]

# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'PARNO')

In [30]:
# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

In [ ]:
# replace anywhere the missing_total_volume is missing (fill in for the outliers since total_volume was computed before)
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

In [49]:
missing_outlier_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County,area_m2,volume_m3,total_volume_m3,residual,missing_total_volume_m3
6372953,ms,UnitedStates_023013221_450317,6.968639,3.512234,USA,"{'xmax': -116.86955377707943, 'xmin': -116.869...","POLYGON ((-11276320.901 3989869.640, -11276308...",Multi,NaN,NaN,None,None,None,218.0,19423173.0,None,NaN,7767377,06073020806,h334,SDGE,RI109,7728229460,San Diego,254.152201,1771.095033,1771.095033,181.835418,1771.095033
6372953,ms,UnitedStates_023013221_450317,6.968639,3.512234,USA,"{'xmax': -116.86955377707943, 'xmin': -116.869...","POLYGON ((-11276320.901 3989869.640, -11276308...",Multi,NaN,NaN,None,None,None,218.0,19423173.0,None,NaN,7767377,06073020806,h334,SDGE,RI109,7728229317,San Diego,254.152201,1771.095033,1771.095033,181.835418,1771.095033
6372953,ms,UnitedStates_023013221_450317,6.968639,3.512234,USA,"{'xmax': -116.86955377707943, 'xmin': -116.869...","POLYGON ((-11276320.901 3989869.640, -11276308...",Multi,NaN,NaN,None,None,None,218.0,19423173.0,None,NaN,7767377,06073020806,h334,SDGE,RI109,7728229510,San Diego,254.152201,1771.095033,1771.095033,181.835418,1771.095033
6372953,ms,UnitedStates_023013221_450317,6.968639,3.512234,USA,"{'xmax': -116.86955377707943, 'xmin': -116.869...","POLYGON ((-11276320.901 3989869.640, -11276308...",Multi,NaN,NaN,None,None,None,218.0,19423173.0,None,NaN,7767377,06073020806,h334,SDGE,RI109,7728229473,San Diego,254.152201,1771.095033,1771.095033,181.835418,1771.095033
6372953,ms,UnitedStates_023013221_450317,6.968639,3.512234,USA,"{'xmax': -116.86955377707943, 'xmin': -116.869...","POLYGON ((-11276320.901 3989869.640, -11276308...",Multi,NaN,NaN,None,None,None,218.0,19423173.0,None,NaN,7767377,06073020806,h334,SDGE,RI109,7728229472,San Diego,254.152201,1771.095033,1771.095033,181.835418,1771.095033


In [32]:
# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

In [34]:
# range of calculated units
missing_outlier_units_pred['unit'].agg(['min','max'])

min     18.0
max    243.0
Name: unit, dtype: float64

In [35]:
missing_outlier_units_pred = missing_outlier_units_pred.drop_duplicates(subset = 'PARNO', keep = 'first')

In [36]:
# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop residual column
multi_complete = multi_complete.drop(['residual'], axis = 1)

# drop the parno column
multi_complete = multi_complete.drop(columns = 'PARNO')

# drop the unnecessary columns of the parcels_res data
parcels_res = parcels_res.drop(columns = ['ADDRESS', 'CITY', 'ZIP', 'unit'])

multi_by_parcel = parcels_res.sjoin(multi_complete, predicate="intersects")


In [50]:
parcels_res.head()

,PARNO,County,Shape_Length,Shape_Area,geometry
7629947,4241720700,San Diego,98.936758,434.984710,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,..."
7630042,4241721300,San Diego,76.116160,347.815384,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,..."
7630043,4241722000,San Diego,106.557535,579.922629,"MULTIPOLYGON Z (((-117.23549 32.79699 0.00000,..."
7630075,4236921300,San Diego,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,..."
7630078,4236220200,San Diego,67.240870,224.046738,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,..."


In [51]:
multi_complete.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,County,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-116.99693 33.29913, -116.99687 33.2...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,San Diego,412.421351,1606.752247,1606.752247,NaN
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-116.98655 33.25268, -116.98620 33.2...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,San Diego,452.260448,1858.907171,1858.907171,NaN
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,San Diego,232.343850,905.530915,905.530915,NaN
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,San Diego,232.343850,905.530915,905.530915,NaN
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-116.37224 33.25284, -116.37224 33.2...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,San Diego,284.844083,1177.213436,1177.213436,NaN


In [52]:
multi_by_parcel.head()

,PARNO,County_left,Shape_Length,Shape_Area,geometry,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,County_right,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3
7630075,4236921300,San Diego,69.352091,238.64133,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,San Diego,140.235708,431.200368,431.200368,NaN
7630075,4236921300,San Diego,69.352091,238.64133,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,San Diego,140.235708,431.200368,431.200368,NaN
7630075,4236921300,San Diego,69.352091,238.64133,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,San Diego,140.235708,431.200368,431.200368,NaN
7630075,4236921300,San Diego,69.352091,238.64133,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,San Diego,140.235708,431.200368,431.200368,NaN
7630075,4236921300,San Diego,69.352091,238.64133,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,San Diego,140.235708,431.200368,431.200368,NaN


### Remove duplicates and make centroids for multi-family homes
Multi-family homes are in polygons. In order to combine the multi-family and single family data we need to make them points. The unit linear regression calculations were completed for buildings so there are multiples of each polygon. We only need one row per parcel to make the centroids. 

In [38]:
# check how many parcel numbers are duplicates
duplicates = multi_by_parcel.pivot_table(index = ['PARNO'], aggfunc = 'size')
print(f"There are ",duplicates[duplicates > 1].sum(), " duplicated parcels.")

There are  3411468  duplicated parcels.


**Remove duplicates so that there aren't duplicate points. The units are aggregated by parcel because they were run on the total_volume.**

In [53]:
# remove duplicates from the parcel number so we don't get multiple points per parcel
multi_by_parcel = multi_by_parcel.drop_duplicates(subset = 'PARNO', keep = 'first')

In [54]:
multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")

In [55]:
## IN ORDER TO CONCAT THE MULTI-FAMILY HOMES WITH SINGLE FAMILY HOMES THEY MUST ALL BE POINTS
# create centroids for each multi residential parcel 
multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid


In [56]:
multi_by_parcel.head()

,PARNO,County_left,Shape_Length,Shape_Area,geometry,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,County_right,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,centroids
7630075,4236921300,San Diego,69.352091,238.641330,MULTIPOLYGON Z (((-11313159.182 3961251.281 0....,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,San Diego,140.235708,431.200368,431.200368,NaN,POINT (-11313153.723 3961239.229)
7630078,4236220200,San Diego,67.240870,224.046738,MULTIPOLYGON Z (((-11313244.013 3962367.862 0....,7349569,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,2.0,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,San Diego,117.733224,320.096607,320.096607,NaN,POINT (-11313238.717 3962356.234)
7630083,4245420100,San Diego,94.646744,497.462992,MULTIPOLYGON Z (((-11311377.586 3963837.438 0....,7316916,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,San Diego,217.808548,1689.639778,1689.639778,NaN,POINT (-11311395.134 3963841.243)
7630084,4245420700,San Diego,92.385658,398.294157,MULTIPOLYGON Z (((-11311356.480 3963757.406 0....,7316886,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,2.0,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,San Diego,113.884017,734.439912,734.439912,NaN,POINT (-11311373.489 3963745.624)
7630116,5670903600,San Diego,298.920714,4397.340123,MULTIPOLYGON Z (((-11298054.295 3947082.794 0....,7106151,osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,40.0,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,San Diego,1895.486015,12525.523059,12525.523059,NaN,POINT (-11298091.345 3947127.302)


In [57]:

# drop the geometry of multi_by_parcel so the centroid becomes the geometry
multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# rename the centroid to geometry
multi_by_parcel = multi_by_parcel.rename(columns = {'centroids' : 'geometry'})

# set the centroid to the geometry
multi_by_parcel_processed = multi_by_parcel.set_geometry('geometry')

In [58]:
multi_by_parcel_processed.head()

,PARNO,County_left,Shape_Length,Shape_Area,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,County_right,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,geometry
7630075,4236921300,San Diego,69.352091,238.641330,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,San Diego,140.235708,431.200368,431.200368,NaN,POINT (-11313153.723 3961239.229)
7630078,4236220200,San Diego,67.240870,224.046738,7349569,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,2.0,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,San Diego,117.733224,320.096607,320.096607,NaN,POINT (-11313238.717 3962356.234)
7630083,4245420100,San Diego,94.646744,497.462992,7316916,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,San Diego,217.808548,1689.639778,1689.639778,NaN,POINT (-11311395.134 3963841.243)
7630084,4245420700,San Diego,92.385658,398.294157,7316886,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,2.0,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,San Diego,113.884017,734.439912,734.439912,NaN,POINT (-11311373.489 3963745.624)
7630116,5670903600,San Diego,298.920714,4397.340123,7106151,osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,40.0,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,San Diego,1895.486015,12525.523059,12525.523059,NaN,POINT (-11298091.345 3947127.302)


### Adjust single-family Zillow data to the SDG&E area and adjust values

In [62]:
# change the CRS back to the Zillow CRS 
multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)

# subset for single family zillow and condos
single_zillow = zillow[zillow['type'] != "Single"].to_crs(zillow.crs)

# keep only single family homes within San Diego County
single_zillow = single_zillow.clip(sdge_shape)

# change all single_zillow to unit = 1
single_zillow['unit'] = 1

# concat the single and multifamily dataframes
complete_zillow = pd.concat([multi_by_parcel_processed, single_zillow], axis = 0)

In [65]:
complete_zillow.head()

,PARNO,County_left,Shape_Length,Shape_Area,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,County_right,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,geometry
7630075,4236921300,San Diego,69.352091,238.641330,7349122.0,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,San Diego,140.235708,431.200368,431.200368,NaN,POINT (-117.25142 32.76728)
7630078,4236220200,San Diego,67.240870,224.046738,7349569.0,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,2.0,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,San Diego,117.733224,320.096607,320.096607,NaN,POINT (-117.25230 32.77765)
7630083,4245420100,San Diego,94.646744,497.462992,7316916.0,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,San Diego,217.808548,1689.639778,1689.639778,NaN,POINT (-117.23320 32.79144)
7630084,4245420700,San Diego,92.385658,398.294157,7316886.0,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,2.0,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,San Diego,113.884017,734.439912,734.439912,NaN,POINT (-117.23297 32.79055)
7630116,5670903600,San Diego,298.920714,4397.340123,7106151.0,osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,40.0,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,San Diego,1895.486015,12525.523059,12525.523059,NaN,POINT (-117.09531 32.63634)


## Final results

## Renaming and saving

In [ ]:
# save the complete Zillow points
complete_zillow_sd = complete_zillow.copy()

In [ ]:
# takes a really long time :,(
complete_zillow_sd.to_file("data/complete_zillow_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 